# EDA completo — dívida PGE (frente macro)

**Frente:** C — dívida ativa inscrita × fatores macroeconômicos.
**Público:** leitura operacional do fluxo aprovado (inventário, três buckets, painel ICMS, correlações, três modelos trimestrais).
**Freeze de referência do dump:** `extracao=2026-03` (`projects/macro/docs/C_divida_macro.md`). Este notebook usa a foto mais recente encontrada na fonte local, ou `EDA_EXTRACAO=YYYY-MM` quando essa variável está definida.

## Como rodar no Mac com o HD montado

O notebook procura os parquet (ou um arquivo DuckDB) a partir de `DATA_ROOT`. Sem essa variável, o caminho é `/Volumes/Meedi_Etore_HD1/CEMEPI/api_ctf/`.

Bash:

```bash
export DATA_ROOT="/Volumes/Meedi_Etore_HD1/CEMEPI/api_ctf/"
```

Fish:

```fish
set -gx DATA_ROOT "/Volumes/Meedi_Etore_HD1/CEMEPI/api_ctf/"
```

Abra o notebook a partir de qualquer pasta do monorepo (o código sobe até achar `shared/cemepi_api`). Kernel sugerido: `ctf-research`, depois de `uv sync` na raiz.

Amostra opcional de CDAs no painel ICMS (não filtra taxa nem arrecadação):

```bash
export EDA_CDA_SAMPLE_N=5000
```

```fish
set -gx EDA_CDA_SAMPLE_N 5000
```

Se o volume não estiver montado, cada seção imprime `fonte não montada`, mostra o próximo passo e segue. Nenhuma contagem, correlação ou coeficiente é preenchido no lugar dos dados.


## O que este arquivo faz

Um único notebook, em sete blocos:

0. Setup e caminhos
1. Inventário e sanidade
2. Mapa `tipo_bruto` → ICMS | IPVA | TAXAS e CSV de-para
3. Descritiva dos três buckets (participação em quantidade, participação em valor, idade da CDA)
4. Painel ICMS em anos (sem filtro `taxa>0` nem `arrecadacao>0`)
5. Correlações de Pearson e Spearman com defasagens 1–3 e valor-p
6. Três modelos trimestrais: logit 0/1, taxa percentual, estoque agregado com macros

Cada bloco termina com uma célula **DoD** que imprime `PASS` ou `FAIL`.

O estoque inscrito mora em `debito` (grão `ID_DEBITO` × `ANO_EXTRACAO` × `MES_EXTRACAO`). O pagamento mora em `arrecadacao`. Essa leitura está no notebook `modelo_dados_api_er_v0.ipynb` e em `projects/macro/README.md`. O dump não traz dívida não inscrita (`docs/references/referencias_grounded.md`, seção C).

Colunas de PII (`NOME_DEVEDOR`, `CPF_DEVEDOR`, `CNPJ_DEVEDOR`) não entram nas projeções.


## 0. Setup + paths


In [1]:
from __future__ import annotations

import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from duckdb import Error as DuckDBError
from numpy.linalg import LinAlgError
from scipy import stats

NB_CWD = Path.cwd().resolve()
MONO = next(
    (
        candidato
        for candidato in [NB_CWD, *NB_CWD.parents]
        if (candidato / "shared" / "cemepi_api").exists()
    ),
    NB_CWD,
)
PLAYGROUND = MONO / "projects" / "macro" / "notebooks" / "playground"
DATA_DIR = PLAYGROUND / "data"
FIG_DIR = MONO / "projects" / "macro" / "output" / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_DATA_ROOT = Path("/Volumes/Meedi_Etore_HD1/CEMEPI/api_ctf/")
PII_COLS = frozenset({"NOME_DEVEDOR", "CPF_DEVEDOR", "CNPJ_DEVEDOR"})
DEBITO_REQ = (
    "ID_DEBITO",
    "CDA_COMPLETA",
    "ANO_EXTRACAO",
    "MES_EXTRACAO",
    "DATA_INSCRICAO",
    "TIPO_DEBITO",
    "VALOR_SEM_HONORARIOS",
)
ARREC_REQ = (
    "ID_DEBITO",
    "CDA_COMPLETA",
    "ANO",
    "MES",
    "VALOR_TOTAL_GARE",
)
BANNED_FILTERS = (
    "taxa_recuperacao > 0",
    "taxa_recuperacao>0",
    "arrecadacao > 0",
    "arrecadacao>0",
    "valor_total_gare > 0",
    "valor_total_gare>0",
    "valor_sem_honorarios > 0",
    "valor_sem_honorarios>0",
)
N_MIN_CORR = 8
CAP_MICRO = 200_000
SEMENTE_AMOSTRA = 202603
ERROS_ETAPA = (
    OSError,
    ValueError,
    KeyError,
    TypeError,
    RuntimeError,
    ArithmeticError,
    DuckDBError,
    LinAlgError,
)

ESTADO: dict[str, object] = {
    "fonte_montada": False,
    "fonte_util": False,
    "tem_debito": False,
    "tem_arrec": False,
    "motivo": "",
    "data_root": DEFAULT_DATA_ROOT,
    "origem_caminho": "default",
    "foto": None,
    "amostra_n": 0,
    "sql_audit": [],
    "fig_buckets": None,
    "depara_csv": None,
    "serie_tri": None,
    "micro": None,
    "macro": None,
    "modelos": {},
    "dod": [],
}


def dod(secao: str, status: str, detalhe: str) -> None:
    """Imprime PASS ou FAIL da seção e guarda o resultado."""
    linha = f"DoD {secao}: {status} — {detalhe}"
    print(linha)
    dods = ESTADO["dod"]
    assert isinstance(dods, list)
    dods.append((secao, status, detalhe))


def resolver_data_root() -> tuple[Path, str]:
    """DATA_ROOT, senão DUMP_ROOT, senão o default do HD Meedi."""
    bruto = os.environ.get("DATA_ROOT", "").strip()
    if bruto:
        return Path(bruto).expanduser(), "DATA_ROOT"
    dump = os.environ.get("DUMP_ROOT", "").strip()
    if dump:
        return Path(dump).expanduser(), "DUMP_ROOT"
    return DEFAULT_DATA_ROOT, "default"


def amostra_n() -> int:
    """Tamanho opcional da amostra de CDAs. Zero significa população."""
    bruto = os.environ.get("EDA_CDA_SAMPLE_N", "").strip()
    if not bruto:
        return 0
    valor = int(bruto)
    if valor < 0:
        raise ValueError("EDA_CDA_SAMPLE_N negativo")
    return valor


def sql_literal(path: Path | str) -> str:
    texto = str(path).replace("'", "''")
    return "'" + texto + "'"


def quote_ident(nome: str) -> str:
    return '"' + nome.replace('"', '""') + '"'


def auditar(sql: str) -> str:
    audit = ESTADO["sql_audit"]
    assert isinstance(audit, list)
    audit.append(sql)
    return sql


def filtros_proibidos(sql: str) -> list[str]:
    compacto = " ".join(sql.lower().split())
    return [item for item in BANNED_FILTERS if item in compacto]


def walk_limitado(raiz: Path):
    """Percorre no máximo cinco níveis e ignora pastas ocultas."""
    base = raiz.resolve()
    for dirpath, dirnames, filenames in os.walk(base):
        relativo = Path(dirpath).resolve().relative_to(base)
        if len(relativo.parts) > 5:
            dirnames.clear()
            continue
        dirnames[:] = [nome for nome in dirnames if not nome.startswith(".")]
        yield Path(dirpath), dirnames, filenames


def achar_diretorios(raiz: Path, nome_tabela: str) -> tuple[list[Path], bool]:
    """Pastas com o nome da tabela e pelo menos um parquet."""
    if not raiz.exists() or not raiz.is_dir():
        return [], False
    achados: list[Path] = []
    truncou = False
    for dirpath, dirnames, filenames in walk_limitado(raiz):
        if dirpath.name.casefold() != nome_tabela.casefold():
            continue
        if any(nome.lower().endswith(".parquet") for nome in filenames):
            achados.append(dirpath)
            dirnames.clear()
        if len(achados) > 40:
            truncou = True
            break
    return achados, truncou


def achar_arquivos(raiz: Path, agulha: str, sufixo: str) -> tuple[list[Path], bool]:
    """Arquivos cujo caminho recente contém a agulha. Trunca em 50."""
    if not raiz.exists() or not raiz.is_dir():
        return [], False
    achados: list[Path] = []
    truncou = False
    for dirpath, _dirnames, filenames in walk_limitado(raiz):
        for nome in filenames:
            caminho = dirpath / nome
            if caminho.suffix.lower() != sufixo:
                continue
            texto = "/".join(part.casefold() for part in caminho.parts[-4:])
            if agulha.casefold() not in texto:
                continue
            achados.append(caminho)
            if len(achados) > 50:
                truncou = True
                return achados, truncou
    return achados, truncou


def diagnosticar_fonte() -> None:
    """Resolve o caminho e relata fonte não montada sem levantar erro."""
    raiz, origem = resolver_data_root()
    ESTADO["data_root"] = raiz
    ESTADO["origem_caminho"] = origem
    ESTADO["amostra_n"] = amostra_n()
    print(f"DATA_ROOT efetivo: {raiz}")
    print(f"origem do caminho: {origem}")
    print(f"monorepo: {MONO}")
    print(f"EDA_CDA_SAMPLE_N: {ESTADO['amostra_n']}")
    if not raiz.exists():
        ESTADO["fonte_montada"] = False
        ESTADO["fonte_util"] = False
        ESTADO["motivo"] = "fonte não montada: caminho inexistente"
        print(ESTADO["motivo"])
        print("Próximos passos:")
        print("1. Monte o volume Meedi_Etore_HD1.")
        print('2. Bash: export DATA_ROOT="/Volumes/Meedi_Etore_HD1/CEMEPI/api_ctf/"')
        print('   Fish: set -gx DATA_ROOT "/Volumes/Meedi_Etore_HD1/CEMEPI/api_ctf/"')
        print("3. Confira parquet de debito e de arrecadacao, ou um .duckdb.")
        print("4. Reexecute este notebook a partir do monorepo.")
        return
    ESTADO["fonte_montada"] = True
    ESTADO["motivo"] = "diretório existe; a seção 1 confere arquivos e colunas"
    print(ESTADO["motivo"])


try:
    diagnosticar_fonte()
except ERROS_ETAPA as exc:
    ESTADO["fonte_montada"] = False
    ESTADO["fonte_util"] = False
    ESTADO["motivo"] = f"fonte não montada: {type(exc).__name__}: {exc}"
    print(ESTADO["motivo"])
    print("Próximos passos: confira a montagem do HD, DATA_ROOT e EDA_CDA_SAMPLE_N.")

CON = duckdb.connect()
print("DuckDB em memória: aberto")

DATA_ROOT efetivo: /Volumes/Meedi_Etore_HD1/CEMEPI/api_ctf
origem do caminho: default
monorepo: /workspace
EDA_CDA_SAMPLE_N: 0
fonte não montada: caminho inexistente
Próximos passos:
1. Monte o volume Meedi_Etore_HD1.
2. Bash: export DATA_ROOT="/Volumes/Meedi_Etore_HD1/CEMEPI/api_ctf/"
   Fish: set -gx DATA_ROOT "/Volumes/Meedi_Etore_HD1/CEMEPI/api_ctf/"
3. Confira parquet de debito e de arrecadacao, ou um .duckdb.
4. Reexecute este notebook a partir do monorepo.
DuckDB em memória: aberto


### DoD 0

`PASS` quando o caminho efetivo foi impresso e, se o volume não existe, a mensagem contém `fonte não montada` e o kernel segue.


In [2]:
caminho = ESTADO["data_root"]
motivo = str(ESTADO["motivo"])
caminho_ok = isinstance(caminho, Path)
ausente_ok = bool(ESTADO["fonte_montada"]) or ("fonte não montada" in motivo)
if caminho_ok and ausente_ok:
    dod("0", "PASS", motivo or "caminho resolvido")
else:
    dod("0", "FAIL", motivo or "caminho não resolvido")

DoD 0: PASS — fonte não montada: caminho inexistente


## 1. Inventário / sanidade


### Contagem e taxa de nulos

A descrição abaixo segue o espírito da análise exploratória de Tukey (1977): olhar a estrutura da tabela antes de qualquer modelo. A citação está em `docs/references/referencias_grounded.md` (seção B).

Para uma coluna $c$ na relação lida:

$$
n = |\{i\}|, \qquad q_{\mathrm{nulo}}(c) = \frac{\#\{i : c_i \ \mathrm{ausente}\}}{n}
$$

$n$ é o número de linhas da view canônica de `debito` (todas as fotos encontradas, não uma amostra). $q_{\mathrm{nulo}}$ é a fração de valores ausentes. As duas quantidades saem de um `COUNT` no DuckDB. Se a fonte não está montada, elas não são substituídas por outro número.

Sanidade adicional, também só com a fonte presente:

- grão declarado: (`ID_DEBITO`, `ANO_EXTRACAO`, `MES_EXTRACAO`);
- fotos distintas e intervalo de anos;
- chaves repetidas nesse grão (o notebook informa a contagem e mantém as linhas).

Colunas exigidas em `debito`: `ID_DEBITO`, `CDA_COMPLETA`, `ANO_EXTRACAO`, `MES_EXTRACAO`, `DATA_INSCRICAO`, `TIPO_DEBITO`, `VALOR_SEM_HONORARIOS`.

Colunas exigidas em `arrecadacao`: `ID_DEBITO`, `CDA_COMPLETA`, `ANO`, `MES`, `VALOR_TOTAL_GARE`.

`ANO`/`MES` da arrecadação é o calendário do pagamento. `ANO_EXTRACAO`/`MES_EXTRACAO` é a foto do estoque. O painel da seção 4 junta os dois pelo ano e pela CDA e registra essa diferença de grão.


In [3]:
def relacao_parquet_dirs(pastas: list[Path]) -> str:
    globs = ", ".join(sql_literal(pasta / "**" / "*.parquet") for pasta in pastas)
    return f"read_parquet([{globs}], union_by_name=true)"


def relacao_parquet_arquivos(arquivos: list[Path]) -> str:
    lista = ", ".join(sql_literal(arquivo) for arquivo in arquivos)
    return f"read_parquet([{lista}], union_by_name=true)"


def nomes_colunas(relacao_sql: str) -> list[str]:
    descricao = CON.execute(f"DESCRIBE SELECT * FROM {relacao_sql}").fetchall()
    return [str(linha[0]) for linha in descricao]


def criar_view_canonica(
    view: str,
    relacao_sql: str,
    requeridas: tuple[str, ...],
) -> list[str]:
    """Projeta só as colunas exigidas. PII fica de fora por omissão."""
    presentes = nomes_colunas(relacao_sql)
    mapa = {nome.upper(): nome for nome in presentes}
    faltantes = [coluna for coluna in requeridas if coluna not in mapa]
    if faltantes:
        return faltantes
    partes = [f"{quote_ident(mapa[coluna])} AS {coluna}" for coluna in requeridas]
    sql = (
        f"CREATE OR REPLACE VIEW {view} AS SELECT "
        + ", ".join(partes)
        + f" FROM {relacao_sql}"
    )
    CON.execute(auditar(sql))
    return []


def escolher_tabela(alias: str, fragmento: str) -> str | None:
    tabelas = CON.execute("SELECT * FROM duckdb_tables()").fetchdf()
    if tabelas.empty:
        return None
    recorte = tabelas[tabelas["database_name"] == alias]
    nomes = recorte["table_name"].astype(str)
    iguais = nomes[nomes.str.casefold() == fragmento.casefold()]
    if iguais.empty:
        iguais = nomes[nomes.str.casefold().str.contains(fragmento.casefold())]
    if iguais.empty:
        return None
    tabela = str(iguais.iloc[0])
    esquema = str(recorte.loc[recorte["table_name"] == tabela, "schema_name"].iloc[0])
    return f"{alias}.{quote_ident(esquema)}.{quote_ident(tabela)}"


def registrar_parquet(nome: str, view: str, colunas: tuple[str, ...]) -> bool:
    pastas, truncou_pasta = achar_diretorios(Path(ESTADO["data_root"]), nome)
    if truncou_pasta:
        print(f"{nome}: varredura de pastas truncada; leitura recusada")
        return False
    if pastas:
        relacao = relacao_parquet_dirs(pastas)
        print(f"{nome}: {len(pastas)} pasta(s) parquet")
        for pasta in pastas:
            print(f"  {pasta}")
    else:
        arquivos, truncou = achar_arquivos(Path(ESTADO["data_root"]), nome, ".parquet")
        if truncou:
            print(f"{nome}: mais de 50 parquet soltos; leitura parcial recusada")
            return False
        if not arquivos:
            print(f"{nome}: nenhum parquet encontrado")
            return False
        relacao = relacao_parquet_arquivos(arquivos)
        print(f"{nome}: {len(arquivos)} arquivo(s) parquet")
    faltantes = criar_view_canonica(view, relacao, colunas)
    if faltantes:
        print(f"{nome}: colunas ausentes: {', '.join(faltantes)}")
        return False
    print(f"{nome}: view {view} pronta")
    return True


def registrar_duckdb() -> None:
    arquivos, truncou = achar_arquivos(Path(ESTADO["data_root"]), ".duckdb", ".duckdb")
    if truncou or not arquivos:
        if truncou:
            print("duckdb: varredura truncada; nenhum banco foi anexado")
        return
    if len(arquivos) > 1:
        print("duckdb: mais de um arquivo; anexe um DATA_ROOT com um único banco")
        for arquivo in arquivos:
            print(f"  {arquivo}")
        return
    banco = arquivos[0]
    CON.execute(auditar(f"ATTACH {sql_literal(banco)} AS src_dump (READ_ONLY)"))
    print(f"duckdb anexado: {banco}")
    if not ESTADO["tem_debito"]:
        relacao = escolher_tabela("src_dump", "debito")
        if relacao is None:
            print("duckdb: tabela debito não encontrada")
        else:
            faltantes = criar_view_canonica("debito_v", relacao, DEBITO_REQ)
            ESTADO["tem_debito"] = not faltantes
            if faltantes:
                print("duckdb debito: colunas ausentes: " + ", ".join(faltantes))
    if not ESTADO["tem_arrec"]:
        relacao = escolher_tabela("src_dump", "arrecadacao")
        if relacao is None:
            print("duckdb: tabela arrecadacao não encontrada")
        else:
            faltantes = criar_view_canonica("arrec_v", relacao, ARREC_REQ)
            ESTADO["tem_arrec"] = not faltantes
            if faltantes:
                print("duckdb arrecadacao: colunas ausentes: " + ", ".join(faltantes))


def foto_pedida() -> tuple[int, int] | None:
    bruto = os.environ.get("EDA_EXTRACAO", "").strip()
    if not bruto:
        return None
    ano_txt, mes_txt = bruto.split("-")
    return int(ano_txt), int(mes_txt)


def carregar_inventario() -> None:
    if not ESTADO["fonte_montada"]:
        print("Inventário não calculado: fonte não montada.")
        print("debito exige: " + ", ".join(DEBITO_REQ))
        print("arrecadacao exige: " + ", ".join(ARREC_REQ))
        print("PII omitida: " + ", ".join(sorted(PII_COLS)))
        return
    ESTADO["tem_debito"] = registrar_parquet("debito", "debito_v", DEBITO_REQ)
    ESTADO["tem_arrec"] = registrar_parquet("arrecadacao", "arrec_v", ARREC_REQ)
    if not ESTADO["tem_debito"] or not ESTADO["tem_arrec"]:
        registrar_duckdb()
    if not ESTADO["tem_debito"]:
        ESTADO["fonte_util"] = False
        print("debito indisponível com o schema exigido")
        return
    colunas = {nome.upper() for nome in nomes_colunas("debito_v")}
    if colunas & PII_COLS:
        raise RuntimeError("view debito_v projetou coluna de PII")
    sql_n = """
        SELECT
          count(*) AS n,
          count(DISTINCT TIPO_DEBITO) AS n_tipos,
          count(DISTINCT ANO_EXTRACAO) AS n_anos,
          min(ANO_EXTRACAO) AS ano_min,
          max(ANO_EXTRACAO) AS ano_max,
          sum((TIPO_DEBITO IS NULL)::INT) AS nulos_tipo,
          sum((TRY_CAST(VALOR_SEM_HONORARIOS AS DOUBLE) IS NULL)::INT) AS nulos_valor,
          sum((TRY_CAST(DATA_INSCRICAO AS DATE) IS NULL)::INT) AS nulos_data
        FROM debito_v
    """
    resumo = CON.execute(auditar(sql_n)).fetchdf()
    print("resumo debito (todas as fotos lidas):")
    print(resumo.to_string(index=False))
    n = int(resumo.loc[0, "n"])
    if n > 0:
        for coluna in ("nulos_tipo", "nulos_valor", "nulos_data"):
            taxa = float(resumo.loc[0, coluna]) / n
            print(f"q_nulo {coluna}: {taxa:.6f}")
    sql_dup = """
        SELECT count(*) AS n_chaves_duplicadas
        FROM (
          SELECT ID_DEBITO, ANO_EXTRACAO, MES_EXTRACAO
          FROM debito_v
          GROUP BY 1, 2, 3
          HAVING count(*) > 1
        )
    """
    duplicadas = CON.execute(auditar(sql_dup)).fetchone()
    print(f"chaves de grão duplicadas: {int(duplicadas[0])}")
    pedida = foto_pedida()
    if pedida is None:
        sql_foto = """
            SELECT ANO_EXTRACAO AS ano, MES_EXTRACAO AS mes
            FROM debito_v
            ORDER BY ANO_EXTRACAO DESC, MES_EXTRACAO DESC
            LIMIT 1
        """
        foto = CON.execute(auditar(sql_foto)).fetchone()
        ESTADO["foto"] = (int(foto[0]), int(foto[1]))
    else:
        ESTADO["foto"] = pedida
    print(
        f"foto usada nas seções 2 e 3: {ESTADO['foto'][0]:04d}-{ESTADO['foto'][1]:02d}"
    )
    ESTADO["fonte_util"] = True


try:
    carregar_inventario()
except ERROS_ETAPA as exc:
    ESTADO["fonte_util"] = False
    print(f"inventário interrompido: {type(exc).__name__}: {exc}")

Inventário não calculado: fonte não montada.
debito exige: ID_DEBITO, CDA_COMPLETA, ANO_EXTRACAO, MES_EXTRACAO, DATA_INSCRICAO, TIPO_DEBITO, VALOR_SEM_HONORARIOS
arrecadacao exige: ID_DEBITO, CDA_COMPLETA, ANO, MES, VALOR_TOTAL_GARE
PII omitida: CNPJ_DEVEDOR, CPF_DEVEDOR, NOME_DEVEDOR


### DoD 1

`PASS` quando a fonte não está montada e o schema exigido foi impresso, ou quando `debito` abriu com as colunas exigidas e a view não contém PII. `FAIL` quando o diretório existe e o schema de `debito` não fecha.


In [4]:
if not ESTADO["fonte_montada"]:
    dod("1", "PASS", "fonte não montada; schema exigido impresso; sem contagens")
elif ESTADO["fonte_util"]:
    dod("1", "PASS", "debito canônico disponível e PII fora da projeção")
else:
    dod("1", "FAIL", "diretório presente sem debito utilizável")

DoD 1: PASS — fonte não montada; schema exigido impresso; sem contagens


## 2. Mapa tipo_bruto → ICMS | IPVA | TAXAS + CSV de-para

### Regra de classificação

`tipo_bruto` é o texto de `TIPO_DEBITO` na foto. A função `classificar_tipo` devolve um bucket e o nome da regra. A mesma regra está no `CASE` SQL.

1. Texto vazio ou nulo → `NAO_MAPEADO`, regra `tipo ausente`.
2. Prefixo `ICMS` (sem distinguir maiúsculas) → `ICMS`, regra `prefixo ICMS`.
3. Prefixo `IPVA` → `IPVA`, regra `prefixo IPVA`.
4. Prefixo `Taxa` → `TAXAS`, regra `prefixo Taxa`.
5. Algum token que começa com `taxa` → `TAXAS`, regra `token Taxa`.
6. Qualquer outro texto → `NAO_MAPEADO`, regra `fora de ICMS, IPVA e TAXAS`.

`Multa Cadastro IPVA` não começa com `IPVA`, então fica em `NAO_MAPEADO`. ITCMD, multas e contribuições também ficam de fora dos três buckets. O resíduo aparece na tabela para a foto não ser reescrita em silêncio.

O teste nominal usa os 43 rótulos completos exibidos em `modelo_dados_api_er_v0.ipynb` para `debito.TIPO_DEBITO` na foto `extracao=2026-03` (44 rótulos naquela célula; um veio truncado como `Multa de Transporte e Trânsito / Emissão de Fo…` e não é reconstruído aqui). O teste compara Python e SQL. A contagem desse teste é contagem de rótulos, não contagem de débitos.

Com a fonte montada, o CSV `projects/macro/notebooks/playground/data/tipo_debito_buckets.csv` recebe `tipo_bruto`, `bucket`, `regra`, `n`, `soma_valor` da foto escolhida. `soma_valor` é a soma de `VALOR_SEM_HONORARIOS`. Sem a fonte, o arquivo não é criado.


In [5]:
BUCKETS = ("ICMS", "IPVA", "TAXAS", "NAO_MAPEADO")
NOMES_2026_03 = (
    "IPVA",
    "ICMS Declarado",
    "Taxa Judiciária",
    "ICMS Autuação",
    "Multa Penal",
    "Multas",
    "ICMS Fecoep",
    "Multa de Nota Fiscal Paulista",
    "Multa Ipca",
    "Multa CBPMESP",
    "Multa Pessoal / Processual",
    "Reposição de Vencimentos",
    "ITCMD Doação Autuação",
    "ITCMD Causa Mortis Declarado",
    "Multa Contratual",
    "Devolução de Qualquer Natureza",
    "ITCMD Causa Mortis Autuação",
    "ICMS Autuação Simples Nacional",
    "Taxas Autuação",
    "ITCMD Doação Declarado",
    "IPVA Autuação",
    "Multa de Outorga",
    "Devolução Nota Fiscal Paulista",
    "Contribuição Previdenciária SPPREV",
    "Multa Daesp",
    "ITBI Doação / Causa Mortis",
    "Despesas Processuais",
    "Taxa de Mandato Judicial",
    "Multa Cadastro IPVA",
    "Cobrança pelo Uso da Água",
    "ICMS Alteração de GIA",
    "Multa Par",
    "Contribuição Previdenciária IPESP",
    "Taxas Declarado",
    "Taxas Autuação Emolumentos",
    "Honorários Periciais",
    "Carteira de Previdência",
    "Estadia Veículo Pátio",
    "Verba Honorária",
    "Diligência de Oficial de Justiça",
    "Multa SAA",
    "Devolução de Taxas de Emolumentos",
    "Repasse",
)


def classificar_tipo(tipo: object) -> tuple[str, str]:
    """Devolve (bucket, regra) para um tipo bruto."""
    if tipo is None:
        return "NAO_MAPEADO", "tipo ausente"
    try:
        if bool(pd.isna(tipo)):
            return "NAO_MAPEADO", "tipo ausente"
    except (TypeError, ValueError):
        pass
    texto = str(tipo).strip()
    if not texto:
        return "NAO_MAPEADO", "tipo ausente"
    chave = texto.casefold()
    if chave.startswith("icms"):
        return "ICMS", "prefixo ICMS"
    if chave.startswith("ipva"):
        return "IPVA", "prefixo IPVA"
    if chave.startswith("taxa"):
        return "TAXAS", "prefixo Taxa"
    tokens = [token.strip(".,;:/") for token in chave.split()]
    if any(token.startswith("taxa") for token in tokens):
        return "TAXAS", "token Taxa"
    return "NAO_MAPEADO", "fora de ICMS, IPVA e TAXAS"


def sql_bucket(coluna: str = "TIPO_DEBITO") -> str:
    chave = f"lower(trim(CAST({coluna} AS VARCHAR)))"
    return (
        "CASE "
        f"WHEN {coluna} IS NULL OR trim(CAST({coluna} AS VARCHAR)) = '' "
        "THEN 'NAO_MAPEADO' "
        f"WHEN {chave} LIKE 'icms%' THEN 'ICMS' "
        f"WHEN {chave} LIKE 'ipva%' THEN 'IPVA' "
        f"WHEN {chave} LIKE 'taxa%' THEN 'TAXAS' "
        f"WHEN {chave} LIKE '% taxa%' OR {chave} LIKE '% taxas%' THEN 'TAXAS' "
        "ELSE 'NAO_MAPEADO' END"
    )


def testar_regras_nominais() -> bool:
    python_df = pd.DataFrame({"tipo_bruto": list(NOMES_2026_03)})
    classes = [classificar_tipo(valor) for valor in python_df["tipo_bruto"]]
    python_df["bucket"] = [item[0] for item in classes]
    python_df["regra"] = [item[1] for item in classes]
    CON.register("tipos_nominais", python_df[["tipo_bruto"]])
    sql = (
        "SELECT tipo_bruto, "
        + sql_bucket("tipo_bruto")
        + " AS bucket FROM tipos_nominais"
    )
    sql_df = CON.execute(auditar(sql)).fetchdf()
    junto = python_df.merge(sql_df, on="tipo_bruto", suffixes=("_py", "_sql"))
    iguais = bool((junto["bucket_py"] == junto["bucket_sql"]).all())
    cobertos = set(python_df["bucket"]).issubset(set(BUCKETS))
    print("teste nominal (rótulos, não população):")
    print(python_df.groupby(["bucket", "regra"]).size().rename("n_rotulos").to_string())
    print(f"python coincide com SQL: {iguais}")
    esperados = {
        "ICMS Declarado": "ICMS",
        "IPVA Autuação": "IPVA",
        "Taxa Judiciária": "TAXAS",
        "Multa Cadastro IPVA": "NAO_MAPEADO",
        "Devolução de Taxas de Emolumentos": "TAXAS",
    }
    pontuais = all(
        classificar_tipo(chave)[0] == bucket for chave, bucket in esperados.items()
    )
    vazio_ok = classificar_tipo(None)[0] == "NAO_MAPEADO"
    print(
        f"casos pontuais: {pontuais}; vazio: {vazio_ok}; n rótulos: {len(NOMES_2026_03)}"
    )
    return iguais and cobertos and pontuais and vazio_ok and len(NOMES_2026_03) == 43


def escrever_depara() -> pd.DataFrame | None:
    if not ESTADO["fonte_util"]:
        print("CSV de-para não escrito: fonte não montada. Sem n nem soma_valor.")
        ESTADO["depara_csv"] = None
        return None
    foto = ESTADO["foto"]
    assert isinstance(foto, tuple)
    ano, mes = int(foto[0]), int(foto[1])
    sql = f"""
        SELECT
          COALESCE(CAST(TIPO_DEBITO AS VARCHAR), '') AS tipo_bruto,
          {sql_bucket()} AS bucket,
          count(*) AS n,
          sum(TRY_CAST(VALOR_SEM_HONORARIOS AS DOUBLE)) AS soma_valor
        FROM debito_v
        WHERE ANO_EXTRACAO = {ano} AND MES_EXTRACAO = {mes}
        GROUP BY 1, 2
        ORDER BY 2, 1
    """
    tabela = CON.execute(auditar(sql)).fetchdf()
    classes = [
        classificar_tipo(valor if valor != "" else None)
        for valor in tabela["tipo_bruto"]
    ]
    tabela["bucket_py"] = [item[0] for item in classes]
    tabela["regra"] = [item[1] for item in classes]
    divergencias = int((tabela["bucket"] != tabela["bucket_py"]).sum())
    print(f"divergências Python/SQL na foto: {divergencias}")
    if divergencias:
        raise RuntimeError("classificação Python e SQL divergiram na foto")
    tabela = tabela[["tipo_bruto", "bucket", "regra", "n", "soma_valor"]]
    destino = DATA_DIR / "tipo_debito_buckets.csv"
    tabela.to_csv(destino, index=False)
    ESTADO["depara_csv"] = destino
    print(f"CSV: {destino}")
    print(tabela.groupby("bucket")[["n", "soma_valor"]].sum().to_string())
    return tabela


REGRAS_OK = False
DEPARA = None
try:
    REGRAS_OK = testar_regras_nominais()
    DEPARA = escrever_depara()
except ERROS_ETAPA as exc:
    REGRAS_OK = False
    print(f"mapa interrompido: {type(exc).__name__}: {exc}")

teste nominal (rótulos, não população):
bucket       regra                     
ICMS         prefixo ICMS                   5
IPVA         prefixo IPVA                   2
NAO_MAPEADO  fora de ICMS, IPVA e TAXAS    30
TAXAS        prefixo Taxa                   5
             token Taxa                     1
python coincide com SQL: True
casos pontuais: True; vazio: True; n rótulos: 43
CSV de-para não escrito: fonte não montada. Sem n nem soma_valor.


### DoD 2

`PASS` quando os 43 rótulos nominais concordam entre Python e SQL e, com a fonte montada, o CSV tem as cinco colunas e a soma de `n` fecha com a foto. Sem a fonte, `PASS` exige que o CSV não tenha sido inventado.


In [6]:
csv_path = ESTADO["depara_csv"]
if not REGRAS_OK:
    dod("2", "FAIL", "regras nominais divergiram ou o teste não rodou")
elif not ESTADO["fonte_util"]:
    if csv_path is None:
        dod(
            "2",
            "PASS",
            "regras nominais ok; CSV omitido porque a fonte não está montada",
        )
    else:
        dod("2", "FAIL", "CSV escrito sem fonte útil")
else:
    quadro = pd.read_csv(csv_path)
    colunas_ok = list(quadro.columns) == [
        "tipo_bruto",
        "bucket",
        "regra",
        "n",
        "soma_valor",
    ]
    foto = ESTADO["foto"]
    assert isinstance(foto, tuple)
    n_foto = CON.execute(
        auditar(
            "SELECT count(*) FROM debito_v "
            f"WHERE ANO_EXTRACAO = {int(foto[0])} AND MES_EXTRACAO = {int(foto[1])}"
        )
    ).fetchone()[0]
    soma_ok = int(quadro["n"].sum()) == int(n_foto)
    if colunas_ok and soma_ok:
        dod(
            "2", "PASS", f"CSV com {len(quadro)} tipos; n soma {int(quadro['n'].sum())}"
        )
    else:
        dod("2", "FAIL", "CSV com colunas ou soma de n fora do contrato")

DoD 2: PASS — regras nominais ok; CSV omitido porque a fonte não está montada


## 3. Descritiva 3 buckets (%qtd vs %valor, idade CDA)

### Participação em quantidade e em valor

Na foto fixada na seção 1, para cada bucket $b$ (incluindo `NAO_MAPEADO`):

$$
p_{q,b} = \frac{n_b}{\sum_b n_b}, \qquad
p_{v,b} = \frac{\sum_{i \in b} v_i}{\sum_i v_i}
$$

$n_b$ é o número de linhas de `debito` na foto. $v_i$ é `VALOR_SEM_HONORARIOS` convertido para número. O denominador das duas frações é a foto inteira, não só os três buckets. A barra do gráfico mostra ICMS, IPVA e TAXAS. A massa `NAO_MAPEADO` sai na tabela, então a soma das três barras pode ficar abaixo de 1.

### Idade da CDA

Na mesma foto, com data de inscrição convertida e mês da extração entre 1 e 12:

$$
\mathrm{idade}_i = \frac{\mathrm{data\_foto} - \mathrm{DATA\_INSCRICAO}_i}{365{,}25}
$$

`data_foto` é o dia 1 do mês da extração. Idade nula (data ilegível ou mês inválido) e idade negativa saem da contagem de percentis e entram em colunas próprias. Esse recorte é de qualidade da data. Ele não remove débito com arrecadação zero.

Os percentis $p_{10}$, $p_{25}$, $p_{50}$, $p_{75}$ e $p_{90}$ são `quantile_cont` no DuckDB, calculados nas idades não nulas e não negativas. O gráfico de caixa usa esses cinco números. As cercas são $p_{10}$ e $p_{90}$. Essa codificação não é a cerca de $1{,}5$ IQR de Tukey (1977), citada em `docs/references/referencias_grounded.md` para outra frente.


### Como ler o gráfico de participação

O eixo horizontal é o bucket (ICMS, IPVA, TAXAS). O eixo vertical é a participação na foto, de 0 a 1. Uma barra é a participação em quantidade. A outra é a participação em valor (`VALOR_SEM_HONORARIOS`). A comparação é entre as duas barras do mesmo bucket e entre buckets. Barras de alturas diferentes no mesmo bucket separam “muitos débitos” de “muito valor”. A figura descreve uma foto do estoque inscrito. Ela não é taxa de recuperação, não é probabilidade de pagamento e não é efeito de política. Se a fonte não está montada, o gráfico é um esqueleto com o aviso `FONTE NÃO MONTADA` e sem percentuais.


In [7]:
def sql_idade() -> str:
    return """
        CASE
          WHEN TRY_CAST(MES_EXTRACAO AS INTEGER) BETWEEN 1 AND 12
           AND TRY_CAST(DATA_INSCRICAO AS DATE) IS NOT NULL
          THEN date_diff(
            'day',
            TRY_CAST(DATA_INSCRICAO AS DATE),
            make_date(
              CAST(ANO_EXTRACAO AS INTEGER),
              CAST(MES_EXTRACAO AS INTEGER),
              1
            )
          ) / 365.25
          ELSE NULL
        END
    """


def figura_participacao(tabela: pd.DataFrame, fonte_util: bool) -> go.Figure:
    tres = tabela[tabela["bucket"].isin(["ICMS", "IPVA", "TAXAS"])].copy()
    if tres.empty:
        fig = go.Figure()
        aviso = (
            "FONTE NÃO MONTADA — sem percentuais"
            if not fonte_util
            else "Nenhum débito ICMS, IPVA ou TAXAS nesta foto"
        )
        fig.update_layout(
            title="Participação dos buckets — esqueleto",
            xaxis_title="Bucket",
            yaxis_title="Participação na foto (0 a 1)",
            annotations=[
                {
                    "text": aviso,
                    "xref": "paper",
                    "yref": "paper",
                    "x": 0.5,
                    "y": 0.5,
                    "showarrow": False,
                }
            ],
        )
        return fig
    longo = tres.melt(
        id_vars=["bucket"],
        value_vars=["p_qtd", "p_valor"],
        var_name="metrica",
        value_name="participacao",
    )
    longo["metrica"] = longo["metrica"].map(
        {
            "p_qtd": "participação em quantidade",
            "p_valor": "participação em valor",
        }
    )
    fig = px.bar(
        longo,
        x="bucket",
        y="participacao",
        color="metrica",
        barmode="group",
        title="Participação na foto: quantidade e valor",
        labels={
            "bucket": "Bucket",
            "participacao": "Participação na foto (0 a 1)",
            "metrica": "Métrica",
        },
    )
    fig.update_layout(yaxis_range=[0, 1], legend_title_text="Métrica")
    return fig


def exportar_figura(fig: go.Figure, stem: str, fonte_util: bool) -> None:
    if not fonte_util:
        print(f"exportação {stem}: omitida (fonte não montada)")
        return
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    png = FIG_DIR / f"{stem}.png"
    svg = FIG_DIR / f"{stem}.svg"
    try:
        fig.write_image(str(png), scale=2)
        fig.write_image(str(svg))
        print(png)
        print(svg)
    except ERROS_ETAPA as exc:
        print(f"write_image falhou ({type(exc).__name__}: {exc})")


def descritiva_buckets() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not ESTADO["fonte_util"]:
        print("Descritiva não calculada: fonte não montada.")
        vazio = pd.DataFrame(columns=["bucket", "n", "soma_valor", "p_qtd", "p_valor"])
        return vazio, pd.DataFrame()
    foto = ESTADO["foto"]
    assert isinstance(foto, tuple)
    ano, mes = int(foto[0]), int(foto[1])
    sql = f"""
        WITH foto AS (
          SELECT
            {sql_bucket()} AS bucket,
            TRY_CAST(VALOR_SEM_HONORARIOS AS DOUBLE) AS valor,
            {sql_idade()} AS idade_anos
          FROM debito_v
          WHERE ANO_EXTRACAO = {ano} AND MES_EXTRACAO = {mes}
        )
        SELECT
          bucket,
          count(*) AS n,
          sum(valor) AS soma_valor,
          sum((idade_anos IS NULL)::INT) AS nulos_idade,
          sum((idade_anos < 0)::INT) AS idade_negativa,
          quantile_cont(idade_anos, 0.10) FILTER (WHERE idade_anos >= 0) AS p10,
          quantile_cont(idade_anos, 0.25) FILTER (WHERE idade_anos >= 0) AS q1,
          quantile_cont(idade_anos, 0.50) FILTER (WHERE idade_anos >= 0) AS mediana,
          quantile_cont(idade_anos, 0.75) FILTER (WHERE idade_anos >= 0) AS q3,
          quantile_cont(idade_anos, 0.90) FILTER (WHERE idade_anos >= 0) AS p90
        FROM foto
        GROUP BY 1
        ORDER BY 1
    """
    tabela = CON.execute(auditar(sql)).fetchdf()
    total_n = float(tabela["n"].sum())
    total_v = float(tabela["soma_valor"].sum())
    tabela["p_qtd"] = tabela["n"] / total_n if total_n else np.nan
    tabela["p_valor"] = tabela["soma_valor"] / total_v if total_v else np.nan
    print(f"foto {ano:04d}-{mes:02d}")
    print(tabela.to_string(index=False))
    print(f"soma p_qtd: {tabela['p_qtd'].sum():.6f}")
    print(f"soma p_valor: {tabela['p_valor'].sum():.6f}")
    return tabela, tabela


TABELA_BUCKETS = pd.DataFrame()
try:
    TABELA_BUCKETS, QUANTIS_IDADE = descritiva_buckets()
except ERROS_ETAPA as exc:
    TABELA_BUCKETS = pd.DataFrame()
    QUANTIS_IDADE = pd.DataFrame()
    print(f"descritiva interrompida: {type(exc).__name__}: {exc}")

FIG_BUCKETS = figura_participacao(TABELA_BUCKETS, bool(ESTADO["fonte_util"]))
ESTADO["fig_buckets"] = FIG_BUCKETS
FIG_BUCKETS.show()
exportar_figura(FIG_BUCKETS, "eda_completo_buckets_share", bool(ESTADO["fonte_util"]))

Descritiva não calculada: fonte não montada.


exportação eda_completo_buckets_share: omitida (fonte não montada)


### Como ler o gráfico de idade

O eixo horizontal é o bucket, inclusive `NAO_MAPEADO` quando a fonte está montada. O eixo vertical é a idade da CDA em anos na data da foto. A caixa marca $p_{25}$, $p_{50}$ e $p_{75}$. A cerca inferior é $p_{10}$ e a superior é $p_{90}$. A comparação é entre buckets na mesma foto. Uma caixa mais alta indica estoque mais antigo naquele bucket. A figura não diz há quanto tempo o débito está sem pagamento, porque a idade usa só a data de inscrição e a data da foto. Sem a fonte, esta figura não é desenhada.


In [8]:
def figura_idade(quantis: pd.DataFrame) -> go.Figure | None:
    if quantis is None or quantis.empty:
        print("Figura de idade omitida: fonte não montada.")
        return None
    fig = go.Figure()
    for linha in quantis.itertuples(index=False):
        fig.add_trace(
            go.Box(
                name=str(linha.bucket),
                q1=[linha.q1],
                median=[linha.mediana],
                q3=[linha.q3],
                lowerfence=[linha.p10],
                upperfence=[linha.p90],
                y=[linha.mediana],
            )
        )
    fig.update_layout(
        title="Idade da CDA na foto (percentis)",
        xaxis_title="Bucket",
        yaxis_title="Idade (anos) na data da foto",
    )
    return fig


FIG_IDADE = figura_idade(QUANTIS_IDADE if ESTADO["fonte_util"] else pd.DataFrame())
if FIG_IDADE is not None:
    FIG_IDADE.show()
    exportar_figura(FIG_IDADE, "eda_completo_idade_cda", True)

Figura de idade omitida: fonte não montada.


### DoD 3

`PASS` quando existe uma figura Plotly de buckets. Com fonte útil, as participações em quantidade somam 1 (tolerância $10^{-6}$) e os três buckets do gráfico estão entre ICMS, IPVA e TAXAS. Sem fonte, o esqueleto precisa trazer o texto `FONTE NÃO MONTADA`.


In [9]:
figura = ESTADO["fig_buckets"]
eh_figura = isinstance(figura, go.Figure)
if not eh_figura:
    dod("3", "FAIL", "figura de buckets ausente")
elif not ESTADO["fonte_util"]:
    textos = []
    for item in figura.layout.annotations or []:
        textos.append(str(getattr(item, "text", item)))
    if any("FONTE NÃO MONTADA" in texto for texto in textos):
        dod("3", "PASS", "esqueleto Plotly sem percentuais")
    else:
        dod("3", "FAIL", "esqueleto sem o aviso de fonte não montada")
else:
    soma = float(TABELA_BUCKETS["p_qtd"].sum())
    buckets_grafico = set(TABELA_BUCKETS["bucket"]) & {"ICMS", "IPVA", "TAXAS"}
    if abs(soma - 1.0) < 1e-6 and buckets_grafico:
        dod("3", "PASS", f"p_qtd soma {soma:.6f}")
    else:
        dod("3", "FAIL", f"participações fora do contrato (soma {soma})")

DoD 3: PASS — esqueleto Plotly sem percentuais


## 4. Painel ICMS (sem filtro taxa>0; amostra opcional de CDA; anos longos)

### Taxa de recuperação e grão longo

O painel é longo: uma linha por CDA e ano, restrita às linhas de `debito` cujo `TIPO_DEBITO` cai em ICMS. Dentro do ano, o estoque é a foto do maior `MES_EXTRACAO` daquela CDA. A arrecadação do ano é a soma de `VALOR_TOTAL_GARE` em `arrecadacao` com o mesmo `CDA_COMPLETA` e o mesmo `ANO`. Ausência de GARE naquele ano entra como arrecadação 0, via `LEFT JOIN`.

$$
\tau_{i,t} =
\begin{cases}
A_{i,t} / E_{i,t} & \text{se } E_{i,t} > 0 \\
\text{ausente} & \text{se } E_{i,t} \le 0 \text{ ou ausente}
\end{cases}
$$

$E_{i,t}$ é o estoque de fim de ano (última foto do ano) e $A_{i,t}$ é a soma dos pagamentos do ano. $\tau=0$ permanece na linha. $\tau$ ausente também permanece. Nenhuma cláusula remove `taxa_recuperacao > 0`, `arrecadacao > 0` ou `valor > 0`.

A amostra opcional (`EDA_CDA_SAMPLE_N`) sorteia CDAs com `USING SAMPLE reservoir` e semente `202603`, antes de olhar $\tau$. Sem a variável, o painel SQL cobre as CDAs ICMS encontradas. A materialização para pandas das colunas numéricas só ocorre até 200 mil linhas, para o logit da seção 6. Acima disso o notebook pede a amostra e não corta os zeros por conta própria.

O painel trimestral, usado nas seções 5 e 6, guarda a última foto de cada CDA dentro do trimestre e soma a arrecadação dos meses daquele trimestre. O agregado trimestral (soma do estoque e soma da arrecadação) cabe em poucas linhas e não depende da amostra.


In [10]:
def sql_views_painel(amostra: int) -> list[str]:
    filtro_amostra = ""
    if amostra > 0:
        filtro_amostra = (
            "AND cda IN (SELECT cda FROM ("
            "SELECT DISTINCT cda FROM icms_base"
            f") USING SAMPLE reservoir({int(amostra)} ROWS) "
            f"REPEATABLE ({SEMENTE_AMOSTRA}))"
        )
    debito_icms = f"""
        CREATE OR REPLACE VIEW icms_base AS
        SELECT
          CAST(CDA_COMPLETA AS VARCHAR) AS cda,
          CAST(ANO_EXTRACAO AS INTEGER) AS ano,
          CAST(MES_EXTRACAO AS INTEGER) AS mes,
          TRY_CAST(VALOR_SEM_HONORARIOS AS DOUBLE) AS estoque,
          TRY_CAST(DATA_INSCRICAO AS DATE) AS data_inscricao
        FROM debito_v
        WHERE {sql_bucket()} = 'ICMS'
          AND CAST(MES_EXTRACAO AS INTEGER) BETWEEN 1 AND 12
    """
    anual = f"""
        CREATE OR REPLACE VIEW icms_ano AS
        SELECT cda, ano, mes, estoque, data_inscricao
        FROM (
          SELECT
            *,
            row_number() OVER (PARTITION BY cda, ano ORDER BY mes DESC) AS rn
          FROM icms_base
          WHERE 1 = 1
          {filtro_amostra}
        )
        WHERE rn = 1
    """
    arrec_ano = """
        CREATE OR REPLACE VIEW arrec_ano AS
        SELECT
          CAST(CDA_COMPLETA AS VARCHAR) AS cda,
          CAST(ANO AS INTEGER) AS ano,
          sum(TRY_CAST(VALOR_TOTAL_GARE AS DOUBLE)) AS arrecadacao
        FROM arrec_v
        GROUP BY 1, 2
    """
    painel_ano = """
        CREATE OR REPLACE VIEW painel_icms_ano AS
        SELECT
          e.cda,
          e.ano,
          e.estoque,
          coalesce(a.arrecadacao, 0) AS arrecadacao,
          CASE
            WHEN e.estoque > 0 THEN coalesce(a.arrecadacao, 0) / e.estoque
            ELSE NULL
          END AS taxa_recuperacao,
          date_diff('day', e.data_inscricao, make_date(e.ano, 12, 31)) / 365.25
            AS idade_anos
        FROM icms_ano e
        LEFT JOIN arrec_ano a
          ON e.cda = a.cda AND e.ano = a.ano
    """
    trimestral = f"""
        CREATE OR REPLACE VIEW icms_tri AS
        SELECT cda, ano, mes, trimestre, estoque, data_inscricao
        FROM (
          SELECT
            *,
            ano * 10 + (((mes - 1) // 3) + 1) AS trimestre,
            row_number() OVER (
              PARTITION BY cda, ano, ((mes - 1) // 3)
              ORDER BY mes DESC
            ) AS rn
          FROM icms_base
          WHERE 1 = 1
          {filtro_amostra}
        )
        WHERE rn = 1
    """
    arrec_tri = """
        CREATE OR REPLACE VIEW arrec_tri AS
        SELECT
          CAST(CDA_COMPLETA AS VARCHAR) AS cda,
          CAST(ANO AS INTEGER) * 10
            + (((CAST(MES AS INTEGER) - 1) // 3) + 1) AS trimestre,
          sum(TRY_CAST(VALOR_TOTAL_GARE AS DOUBLE)) AS arrecadacao
        FROM arrec_v
        WHERE CAST(MES AS INTEGER) BETWEEN 1 AND 12
        GROUP BY 1, 2
    """
    painel_tri = """
        CREATE OR REPLACE VIEW painel_icms_tri AS
        SELECT
          e.cda,
          e.trimestre,
          e.estoque,
          coalesce(a.arrecadacao, 0) AS arrecadacao,
          CASE
            WHEN e.estoque > 0 THEN coalesce(a.arrecadacao, 0) / e.estoque
            ELSE NULL
          END AS taxa_recuperacao,
          date_diff('day', e.data_inscricao, make_date(e.ano, e.mes, 1)) / 365.25
            AS idade_anos
        FROM icms_tri e
        LEFT JOIN arrec_tri a
          ON e.cda = a.cda AND e.trimestre = a.trimestre
    """
    agregado = """
        CREATE OR REPLACE VIEW serie_icms_tri AS
        SELECT
          trimestre,
          sum(estoque) AS estoque,
          sum(arrecadacao) AS arrecadacao,
          CASE
            WHEN sum(CASE WHEN estoque > 0 THEN estoque ELSE 0 END) > 0
            THEN sum(arrecadacao)
              / sum(CASE WHEN estoque > 0 THEN estoque ELSE 0 END)
            ELSE NULL
          END AS taxa
        FROM painel_icms_tri
        GROUP BY 1
        ORDER BY 1
    """
    return [
        debito_icms,
        anual,
        arrec_ano,
        painel_ano,
        trimestral,
        arrec_tri,
        painel_tri,
        agregado,
    ]


SQL_PAINEL = sql_views_painel(int(ESTADO["amostra_n"]))
PROIBIDOS = []
for comando in SQL_PAINEL:
    auditar(comando)
    PROIBIDOS.extend(filtros_proibidos(comando))
print(f"comandos SQL do painel: {len(SQL_PAINEL)}")
print(f"filtros proibidos encontrados: {PROIBIDOS}")


def materializar_painel() -> None:
    if not (ESTADO["fonte_util"] and ESTADO["tem_arrec"]):
        print("Painel não materializado: fonte não montada ou arrecadacao ausente.")
        ESTADO["serie_tri"] = None
        ESTADO["micro"] = None
        return
    for comando in SQL_PAINEL:
        CON.execute(comando)
    diag = CON.execute(
        auditar(
            """
            SELECT
              count(*) AS n,
              count(DISTINCT ano) AS n_anos,
              min(ano) AS ano_min,
              max(ano) AS ano_max,
              sum((arrecadacao = 0)::INT) AS n_arrec_zero,
              sum((taxa_recuperacao = 0)::INT) AS n_taxa_zero,
              sum((taxa_recuperacao IS NULL)::INT) AS n_taxa_nula
            FROM painel_icms_ano
            """
        )
    ).fetchdf()
    print("painel ICMS anual:")
    print(diag.to_string(index=False))
    serie = CON.execute("SELECT * FROM serie_icms_tri").fetchdf()
    ESTADO["serie_tri"] = serie
    print(f"trimestres agregados: {len(serie)}")
    n_tri = int(CON.execute("SELECT count(*) FROM painel_icms_tri").fetchone()[0])
    print(f"linhas CDA x trimestre: {n_tri}")
    if n_tri > CAP_MICRO and int(ESTADO["amostra_n"]) == 0:
        print(
            "Microdados não copiados para pandas: acima de "
            f"{CAP_MICRO} linhas. Defina EDA_CDA_SAMPLE_N para uma amostra "
            "aleatória de CDAs. A amostra não filtra taxa nem arrecadação."
        )
        ESTADO["micro"] = None
        return
    micro = CON.execute(
        """
        SELECT trimestre, estoque, arrecadacao, taxa_recuperacao, idade_anos
        FROM painel_icms_tri
        """
    ).fetchdf()
    ESTADO["micro"] = micro
    print(f"microdados numéricos: {micro.shape[0]} linhas, sem identificador de CDA")


try:
    materializar_painel()
except ERROS_ETAPA as exc:
    ESTADO["serie_tri"] = None
    ESTADO["micro"] = None
    print(f"painel interrompido: {type(exc).__name__}: {exc}")

comandos SQL do painel: 8
filtros proibidos encontrados: []
Painel não materializado: fonte não montada ou arrecadacao ausente.


### DoD 4

`PASS` quando o SQL do painel não contém filtro de taxa, arrecadação ou valor estritamente positivo. Com fonte e arrecadação, o diagnóstico anual precisa ter sido impresso. Sem fonte, o SQL ainda é auditado e o painel não recebe números fictícios.


In [11]:
if PROIBIDOS:
    dod("4", "FAIL", "SQL com filtro positivo: " + ", ".join(PROIBIDOS))
elif not ESTADO["fonte_util"] or not ESTADO["tem_arrec"]:
    serie = ESTADO["serie_tri"]
    if serie is None:
        dod(
            "4",
            "PASS",
            "SQL auditado; painel sem números porque a fonte não está pronta",
        )
    else:
        dod("4", "FAIL", "série preenchida sem fonte útil")
else:
    dod("4", "PASS", "painel ICMS materializado sem filtro taxa>0 ou arrecadacao>0")

DoD 4: PASS — SQL auditado; painel sem números porque a fonte não está pronta


## 5. Correlações + lags 1–3 (Pearson/Spearman + p)

### Pearson, Spearman e defasagem

Na série trimestral agregada do ICMS, para duas colunas $x$ e $y$ e defasagem $k \in \{0,1,2,3\}$:

$$
r_k = \frac{\sum_t (x_t-\bar x)(y_{t-k}-\bar y)}
{\sqrt{\sum_t (x_t-\bar x)^2 \, \sum_t (y_{t-k}-\bar y)^2}}
$$

$r_k$ é o coeficiente produto-momento entre $x_t$ e $y_{t-k}$, nas datas em que os dois valores existem. O coeficiente de Spearman é o mesmo produto-momento aplicado aos postos. Os dois e os valores-p bicaudais saem de `scipy.stats.pearsonr` e `scipy.stats.spearmanr`.

`docs/references/referencias_grounded.md` ainda não registra a fonte primária desses dois coeficientes. Este notebook escreve a fórmula e não inventa a referência.

O par só entra na tabela quando restam pelo menos 8 trimestres depois do alinhamento e da defasagem. Abaixo disso a linha não é criada: não há coeficiente de preenchimento. Em $k=0$ a tabela guarda cada par não ordenado uma vez ($x < y$ no nome da coluna). Em $k>0$ entra também a autocorrelação ($x$ com $x$ defasado).

Hyndman e Athanasopoulos (2021, *Forecasting: principles and practice*, https://otexts.com/fpp3/, citado na seção B das referências grounded) separam associação de previsão. Um $r_k$ alto descreve co-movimento nesta série. Ele não identifica efeito de Selic, IPCA ou REFIS. Cointegração fica para depois do painel estável (seção C das mesmas referências).

Séries macro entram só se houver parquet local com colunas `selic` e `ipca` (amostra `c2_macro_sample.parquet` do notebook `02_macro_bcb_sample.ipynb`; SGS 432 e 433). Este notebook não consulta a API do BCB.


### Como ler o mapa de correlação

Quando a série existe, o mapa mostra o Pearson em $k=0$. O eixo horizontal e o eixo vertical são os nomes das séries (estoque, arrecadação, taxa e, se existirem, Selic e IPCA). A cor é $r_0$ no intervalo de $-1$ a $1$. A comparação é entre pares na mesma janela trimestral, com o mesmo $n$ mínimo de 8. Caselas ausentes não receberam coeficiente. O mapa não é uma matriz de causalidade. Sem linhas estimadas, o mapa não é desenhado.


In [12]:
def localizar_macro() -> Path | None:
    raiz = Path(ESTADO["data_root"])
    candidatos: list[Path] = []
    if raiz.exists():
        arquivos, _truncou = achar_arquivos(raiz, "macro", ".parquet")
        candidatos.extend(arquivos)
        selic, _ = achar_arquivos(raiz, "selic", ".parquet")
        candidatos.extend(selic)
    dump = os.environ.get("DUMP_ROOT", "").strip()
    if dump:
        candidatos.extend(Path(dump).glob("extracao=*/samples/c2_macro_sample.parquet"))
    vistos: list[Path] = []
    for caminho in candidatos:
        if caminho not in vistos and caminho.exists():
            vistos.append(caminho)
    if not vistos:
        print(
            "Série macro local ausente. Selic = BCB SGS 432; IPCA = BCB SGS 433. "
            "Grave o parquet com notebooks/02_macro_bcb_sample.ipynb. "
            "Este EDA não baixa a API."
        )
        return None
    print(f"macro local: {vistos[0]}")
    return vistos[0]


def agregar_macro(caminho: Path) -> pd.DataFrame | None:
    quadro = pd.read_parquet(caminho)
    if isinstance(quadro.index, pd.DatetimeIndex):
        quadro = quadro.reset_index()
    colunas = {nome.casefold(): nome for nome in quadro.columns}
    if "selic" not in colunas or "ipca" not in colunas:
        print(
            "macro sem colunas selic e ipca: " + ", ".join(quadro.columns.astype(str))
        )
        return None
    coluna_data = None
    for nome in quadro.columns:
        if nome.casefold() in {"date", "data", "index"}:
            coluna_data = nome
            break
    if coluna_data is None:
        print("macro sem coluna de data")
        return None
    quadro = quadro.rename(
        columns={
            coluna_data: "data",
            colunas["selic"]: "selic",
            colunas["ipca"]: "ipca",
        }
    )
    quadro["data"] = pd.to_datetime(quadro["data"], errors="coerce")
    quadro = quadro.dropna(subset=["data"])
    quadro["trimestre"] = quadro["data"].dt.year * 10 + quadro["data"].dt.quarter
    agregado = quadro.groupby("trimestre", as_index=False)[["selic", "ipca"]].mean()
    print(
        "macro trimestral: média dos valores observados dentro do trimestre. "
        "A média do IPCA mensal (SGS 433) não é a inflação composta do trimestre."
    )
    print(f"trimestres macro: {len(agregado)}")
    return agregado


def serie_para_correlacao() -> pd.DataFrame | None:
    serie = ESTADO["serie_tri"]
    if not isinstance(serie, pd.DataFrame) or serie.empty:
        print("Correlação não calculada: série trimestral ausente.")
        return None
    quadro = serie.copy()
    caminho = localizar_macro()
    if caminho is not None:
        macro = agregar_macro(caminho)
        ESTADO["macro"] = macro
        if macro is not None:
            quadro = quadro.merge(macro, on="trimestre", how="left")
    else:
        ESTADO["macro"] = None
    return quadro


def correlacionar(serie: pd.DataFrame) -> pd.DataFrame:
    colunas = [nome for nome in serie.columns if nome != "trimestre"]
    base = serie.sort_values("trimestre").reset_index(drop=True)
    registros: list[dict[str, object]] = []
    for k in (0, 1, 2, 3):
        defasado = base[colunas].shift(k)
        for x in colunas:
            alvos = colunas if k else [y for y in colunas if y > x]
            for y in alvos:
                par = pd.concat(
                    [base[x], defasado[y]],
                    axis=1,
                    keys=["x", "y"],
                ).dropna()
                n = len(par)
                if n < N_MIN_CORR:
                    print(f"sem estimativa: {x} vs {y} lag {k} (n={n} < {N_MIN_CORR})")
                    continue
                r_p, p_p = stats.pearsonr(par["x"], par["y"])
                r_s, p_s = stats.spearmanr(par["x"], par["y"])
                registros.append(
                    {
                        "x": x,
                        "y": y,
                        "lag": k,
                        "n": n,
                        "pearson_r": float(r_p),
                        "pearson_p": float(p_p),
                        "spearman_r": float(r_s),
                        "spearman_p": float(p_s),
                    }
                )
    return pd.DataFrame(registros)


def figura_correlacao(tabela: pd.DataFrame) -> go.Figure | None:
    lag0 = tabela[tabela["lag"] == 0] if not tabela.empty else tabela
    if lag0.empty:
        print("Mapa de correlação omitido: nenhum par com n suficiente.")
        return None
    nomes = sorted(set(lag0["x"]).union(set(lag0["y"])))
    matriz = pd.DataFrame(np.eye(len(nomes)), index=nomes, columns=nomes)
    for linha in lag0.itertuples(index=False):
        matriz.loc[linha.x, linha.y] = linha.pearson_r
        matriz.loc[linha.y, linha.x] = linha.pearson_r
    fig = px.imshow(
        matriz,
        zmin=-1,
        zmax=1,
        color_continuous_scale="RdBu_r",
        title="Pearson contemporâneo (lag 0) — série trimestral ICMS",
        labels={"x": "Série", "y": "Série", "color": "r de Pearson"},
    )
    return fig


CORR = pd.DataFrame()
try:
    SERIE_CORR = serie_para_correlacao()
    if SERIE_CORR is not None:
        CORR = correlacionar(SERIE_CORR)
        if CORR.empty:
            print(
                "Nenhum par atingiu o n mínimo. Tabela vazia, sem coeficiente fictício."
            )
        else:
            print(CORR.to_string(index=False))
        FIG_CORR = figura_correlacao(CORR)
        if FIG_CORR is not None:
            FIG_CORR.show()
            exportar_figura(FIG_CORR, "eda_completo_corr_lag0", True)
except ERROS_ETAPA as exc:
    CORR = pd.DataFrame()
    print(f"correlação interrompida: {type(exc).__name__}: {exc}")

Correlação não calculada: série trimestral ausente.


### DoD 5

`PASS` quando, sem série, a tabela fica vazia, ou quando cada linha estimada tem Pearson, Spearman, valor-p e $n \ge 8$. `FAIL` quando aparece linha incompleta ou $n$ abaixo do mínimo.


In [13]:
if not isinstance(CORR, pd.DataFrame):
    dod("5", "FAIL", "tabela de correlação não foi criada")
elif CORR.empty:
    dod("5", "PASS", "nenhum coeficiente emitido (fonte ausente ou n < 8)")
else:
    colunas = {"pearson_r", "pearson_p", "spearman_r", "spearman_p", "n", "lag"}
    completa = (
        colunas.issubset(CORR.columns) and not CORR[list(colunas)].isna().any().any()
    )
    lags_ok = set(CORR["lag"]).issubset({0, 1, 2, 3})
    n_ok = bool((CORR["n"] >= N_MIN_CORR).all())
    if completa and lags_ok and n_ok:
        dod("5", "PASS", f"{len(CORR)} pares estimados com scipy")
    else:
        dod("5", "FAIL", "pares estimados fora do contrato")

DoD 5: PASS — nenhum coeficiente emitido (fonte ausente ou n < 8)


## 6. Três modelos trimestrais: logit 0/1; taxa %; estoque agregado + macros

Os três ajustes usam o trimestre. Eles só devolvem coeficiente quando o estimador roda. Caso contrário o retorno é `bloqueado` e `params` fica vazio. Isso cobre fonte ausente, amostra micro não materializada, uma só classe no logit, ou macro inexistente.

### Logit 0/1

Para a CDA $i$ no trimestre $q$:

$$
Y_{i,q} = \mathbf{1}\{A_{i,q} > 0\}, \qquad
\Pr(Y_{i,q}=1 \mid x_{i,q}) = \Lambda(\beta_0 + x_{i,q}^{\top}\beta), \qquad
\Lambda(z) = (1+e^{-z})^{-1}
$$

$A_{i,q}$ é a arrecadação do trimestre. $Y=0$ permanece na amostra. O vetor $x$ tem a idade em anos e $\log E_{i,q}$. O logaritmo existe quando $E_{i,q}>0$. Linhas sem estoque positivo saem desta regressão, e a contagem dessa exclusão é impressa. A exclusão é o suporte do logaritmo, não um filtro de taxa de recuperação positiva. O ajuste é `statsmodels.Logit`. A fonte primária do logit não está em `docs/references/referencias_grounded.md`; a fórmula acima é a especificação usada aqui.

### Taxa percentual

$$
\tau^{\%}_{i,q} = 100 \times \frac{A_{i,q}}{E_{i,q}} \quad (E_{i,q}>0)
$$

O segundo modelo é MQO de $\tau^{\%}$ sobre a mesma idade e o mesmo log do estoque. $\tau^{\%}=0$ permanece. Linhas com estoque não positivo ficam de fora porque $\tau$ não está definido, com a contagem impressa.

### Estoque agregado e macros

$$
E_q = \alpha_0 + \alpha_1 \mathrm{Selic}_q + \alpha_2 \mathrm{IPCA}_q + \varepsilon_q
$$

$E_q$ é a soma do estoque ICMS no fim do trimestre (última foto de cada CDA dentro do trimestre). Selic e IPCA são médias trimestrais do parquet local (SGS 432 e 433; seção C das referências grounded). O MQO exige pelo menos 8 trimestres com as três séries. O coeficiente descreve co-movimento linear nessa janela. Ele não é efeito causal e não substitui o teste de cointegração previsto no plano C (`projects/macro/docs/C_divida_macro.md`).


In [14]:
def preparar_micro(df: pd.DataFrame | None) -> pd.DataFrame | None:
    if df is None or df.empty:
        return None
    quadro = df.copy()
    quadro["y_recup"] = (quadro["arrecadacao"].fillna(0) > 0).astype(int)
    quadro["log_estoque"] = np.log(quadro["estoque"].where(quadro["estoque"] > 0))
    quadro["taxa_pct"] = np.where(
        quadro["estoque"] > 0,
        100.0 * quadro["arrecadacao"].fillna(0) / quadro["estoque"],
        np.nan,
    )
    return quadro


def ajustar_logit(df: pd.DataFrame | None) -> dict[str, object]:
    quadro = preparar_micro(df)
    if quadro is None:
        return {"status": "bloqueado", "motivo": "sem painel micro", "params": None}
    n_total = len(quadro)
    n_zero = int((quadro["y_recup"] == 0).sum())
    usado = quadro.dropna(subset=["y_recup", "idade_anos", "log_estoque"])
    print(f"logit: n={n_total}; y=0: {n_zero}; usados: {len(usado)}")
    if usado["y_recup"].nunique() < 2 or len(usado) < N_MIN_CORR:
        return {
            "status": "bloqueado",
            "motivo": "amostra logit sem duas classes ou n < 8",
            "params": None,
            "n": len(usado),
        }
    exogenas = sm.add_constant(usado[["idade_anos", "log_estoque"]], has_constant="add")
    try:
        modelo = sm.Logit(usado["y_recup"], exogenas).fit(disp=False)
    except (ValueError, RuntimeError, ArithmeticError, LinAlgError) as exc:
        return {
            "status": "bloqueado",
            "motivo": f"{type(exc).__name__}: {exc}",
            "params": None,
        }
    print(modelo.summary().as_text())
    return {
        "status": "estimado",
        "motivo": "Logit statsmodels",
        "params": {chave: float(valor) for chave, valor in modelo.params.items()},
        "n": int(modelo.nobs),
    }


def ajustar_taxa(df: pd.DataFrame | None) -> dict[str, object]:
    quadro = preparar_micro(df)
    if quadro is None:
        return {"status": "bloqueado", "motivo": "sem painel micro", "params": None}
    n_total = len(quadro)
    usado = quadro.dropna(subset=["taxa_pct", "idade_anos", "log_estoque"])
    n_zero = int((usado["taxa_pct"] == 0).sum()) if not usado.empty else 0
    print(
        f"taxa %: n={n_total}; usados: {len(usado)}; taxa_pct=0 entre usados: {n_zero}"
    )
    if len(usado) < N_MIN_CORR:
        return {
            "status": "bloqueado",
            "motivo": "n < 8 na regressão da taxa",
            "params": None,
            "n": len(usado),
        }
    exogenas = sm.add_constant(usado[["idade_anos", "log_estoque"]], has_constant="add")
    try:
        modelo = sm.OLS(usado["taxa_pct"], exogenas).fit()
    except (ValueError, RuntimeError, ArithmeticError, LinAlgError) as exc:
        return {
            "status": "bloqueado",
            "motivo": f"{type(exc).__name__}: {exc}",
            "params": None,
        }
    print(modelo.summary().as_text())
    return {
        "status": "estimado",
        "motivo": "MQO statsmodels",
        "params": {chave: float(valor) for chave, valor in modelo.params.items()},
        "n": int(modelo.nobs),
    }


def ajustar_estoque_macro(
    serie: pd.DataFrame | None, macro: pd.DataFrame | None
) -> dict[str, object]:
    if not isinstance(serie, pd.DataFrame) or serie.empty:
        return {"status": "bloqueado", "motivo": "sem série de estoque", "params": None}
    if not isinstance(macro, pd.DataFrame) or macro.empty:
        return {"status": "bloqueado", "motivo": "sem Selic/IPCA local", "params": None}
    junto = serie.merge(macro, on="trimestre", how="inner")
    usado = junto.dropna(subset=["estoque", "selic", "ipca"])
    print(f"estoque+macro: trimestres sobrepostos: {len(usado)}")
    if len(usado) < N_MIN_CORR:
        return {
            "status": "bloqueado",
            "motivo": "menos de 8 trimestres com estoque, Selic e IPCA",
            "params": None,
            "n": len(usado),
        }
    exogenas = sm.add_constant(usado[["selic", "ipca"]], has_constant="add")
    try:
        modelo = sm.OLS(usado["estoque"], exogenas).fit()
    except (ValueError, RuntimeError, ArithmeticError, LinAlgError) as exc:
        return {
            "status": "bloqueado",
            "motivo": f"{type(exc).__name__}: {exc}",
            "params": None,
        }
    print(modelo.summary().as_text())
    return {
        "status": "estimado",
        "motivo": "MQO do estoque agregado",
        "params": {chave: float(valor) for chave, valor in modelo.params.items()},
        "n": int(modelo.nobs),
    }


def rodar_modelos() -> dict[str, dict[str, object]]:
    micro = ESTADO["micro"]
    serie = ESTADO["serie_tri"]
    macro = ESTADO["macro"]
    if macro is None and (not isinstance(serie, pd.DataFrame) or serie.empty):
        localizar_macro()
    saida = {
        "logit": ajustar_logit(micro if isinstance(micro, pd.DataFrame) else None),
        "taxa_pct": ajustar_taxa(micro if isinstance(micro, pd.DataFrame) else None),
        "estoque_macro": ajustar_estoque_macro(
            serie if isinstance(serie, pd.DataFrame) else None,
            macro if isinstance(macro, pd.DataFrame) else None,
        ),
    }
    for nome, resultado in saida.items():
        print(f"modelo {nome}: {resultado['status']} — {resultado['motivo']}")
    return saida


try:
    ESTADO["modelos"] = rodar_modelos()
except ERROS_ETAPA as exc:
    ESTADO["modelos"] = {}
    print(f"modelos interrompidos: {type(exc).__name__}: {exc}")

Série macro local ausente. Selic = BCB SGS 432; IPCA = BCB SGS 433. Grave o parquet com notebooks/02_macro_bcb_sample.ipynb. Este EDA não baixa a API.
modelo logit: bloqueado — sem painel micro
modelo taxa_pct: bloqueado — sem painel micro
modelo estoque_macro: bloqueado — sem série de estoque


### DoD 6

`PASS` quando os três nomes (`logit`, `taxa_pct`, `estoque_macro`) existem, cada status é `estimado` ou `bloqueado`, e `bloqueado` não traz coeficiente. `estimado` precisa trazer o dicionário devolvido pelo ajuste.


In [15]:
modelos = ESTADO["modelos"]
nomes = ("logit", "taxa_pct", "estoque_macro")
if not isinstance(modelos, dict) or any(nome not in modelos for nome in nomes):
    dod("6", "FAIL", "faltou algum dos três modelos")
else:
    ok = True
    for nome in nomes:
        item = modelos[nome]
        status = item.get("status")
        params = item.get("params")
        if status == "bloqueado" and params is None:
            continue
        if status == "estimado" and isinstance(params, dict) and params:
            continue
        ok = False
        print(f"contrato quebrado em {nome}: {item}")
    if ok:
        dod("6", "PASS", "três modelos com status honesto")
    else:
        dod(
            "6",
            "FAIL",
            "algum modelo estimado sem coeficiente, ou bloqueado com coeficiente",
        )

DoD 6: PASS — três modelos com status honesto


## Fecho

A célula abaixo lista o DoD das seções 0 a 6. Com o HD desmontado, o resultado esperado é `PASS` em todas: regras nominais e SQL do painel foram conferidos, e nenhuma tabela analítica foi preenchida com número de estoque, participação, correlação ou coeficiente. Com o HD montado, as seções 1 a 6 passam a imprimir as contagens lidas do parquet ou do DuckDB.


In [16]:
quadro_dod = pd.DataFrame(ESTADO["dod"], columns=["secao", "status", "detalhe"])
print(quadro_dod.to_string(index=False))
falhas = int((quadro_dod["status"] == "FAIL").sum())
print(f"falhas DoD: {falhas}")
if falhas:
    raise RuntimeError(f"DoD com {falhas} FAIL")

secao status                                                         detalhe
    0   PASS                          fonte não montada: caminho inexistente
    1   PASS       fonte não montada; schema exigido impresso; sem contagens
    2   PASS regras nominais ok; CSV omitido porque a fonte não está montada
    3   PASS                                esqueleto Plotly sem percentuais
    4   PASS SQL auditado; painel sem números porque a fonte não está pronta
    5   PASS             nenhum coeficiente emitido (fonte ausente ou n < 8)
    6   PASS                                 três modelos com status honesto
falhas DoD: 0
